# 🧠 Google MuRIL v2: 6-Class Multilingual Cyberbullying Fine-Tuning

**Project:** Cyberbullying Detection & Explainability System (Multilingual MuRIL v2)  
**Goal:** Fine-tune Google MuRIL across combined **English (Kaggle)** and **Hinglish (BullyExplain)** datasets as a unified 6-class demographic classifier.

### Taxonomy (6 Classes):
1. `age`
2. `ethnicity`
3. `gender`
4. `religion`
5. `other_cyberbullying`
6. `not_cyberbullying`

In [ ]:
import os
import sys
import time
import random
import json
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from torch.optim import AdamW
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

warnings.filterwarnings('ignore')

# Set Seeds
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f" PyTorch Device: {device}")
if torch.cuda.is_available():
    print(f" GPU Model: {torch.cuda.get_device_name(0)}")

## 1. Load Combined Multilingual Datasets (6-Class)

In [ ]:
data_dir = os.path.join('..', 'data', 'processed')
train_df = pd.read_parquet(os.path.join(data_dir, 'combined_train.parquet'))
val_df = pd.read_parquet(os.path.join(data_dir, 'combined_val.parquet'))
test_df = pd.read_parquet(os.path.join(data_dir, 'combined_test.parquet'))

print(f"Combined Splits: Train={len(train_df):,}, Val={len(val_df):,}, Test={len(test_df):,}")
print("\n--- Training Class Distribution ---")
print(train_df['cyberbullying_type'].value_counts())

LABEL_MAP = {
    'age': 0,
    'ethnicity': 1,
    'gender': 2,
    'not_cyberbullying': 3,
    'other_cyberbullying': 4,
    'religion': 5
}
ID_TO_LABEL = {v: k for k, v in LABEL_MAP.items()}
CLASS_NAMES = [ID_TO_LABEL[i] for i in range(len(LABEL_MAP))]

## 2. Load Local MuRIL Base Model & Tokenizer

In [ ]:
MODEL_PATH = os.path.join('..', 'models', 'muril_base_safetensors')
print(f"Loading MuRIL base tokenizer and weights from {MODEL_PATH}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_PATH,
    num_labels=len(LABEL_MAP),
    id2label=ID_TO_LABEL,
    label2id=LABEL_MAP
).to(device)

MAX_LEN = 128
print(f" Model loaded onto {device} with {len(LABEL_MAP)} output heads.")

## 3. Dataset Loader & PyTorch DataLoaders

In [ ]:
class CyberbullyingDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long)
        }

BATCH_SIZE = 32
train_dataset = CyberbullyingDataset(train_df['cleaned_text'].values, [LABEL_MAP[t] for t in train_df['cyberbullying_type']], tokenizer, max_len=MAX_LEN)
val_dataset = CyberbullyingDataset(val_df['cleaned_text'].values, [LABEL_MAP[t] for t in val_df['cyberbullying_type']], tokenizer, max_len=MAX_LEN)
test_dataset = CyberbullyingDataset(test_df['cleaned_text'].values, [LABEL_MAP[t] for t in test_df['cyberbullying_type']], tokenizer, max_len=MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"DataLoaders created -> Train Batches: {len(train_loader)}, Val Batches: {len(val_loader)}, Test Batches: {len(test_loader)}")

## 4. Fine-Tuning Optimization Loop

In [ ]:
EPOCHS = 2
LR = 2e-5
total_steps = len(train_loader) * EPOCHS

optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps)
criterion = nn.CrossEntropyLoss()
scaler = torch.amp.GradScaler('cuda', enabled=torch.cuda.is_available())

def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
                out = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = criterion(out.logits, labels)
            total_loss += loss.item()
            preds = torch.argmax(out.logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
    acc = accuracy_score(all_labels, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average='macro', zero_division=0)
    return total_loss / len(loader), acc, prec, rec, f1, all_preds, all_labels

print("=" * 70)
print("STARTING TRAINING")
print("=" * 70)

best_f1 = 0.0
output_dir = os.path.join('..', 'models', 'muril_cyberbullying_v2')
os.makedirs(output_dir, exist_ok=True)

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    print(f"\n--- Epoch {epoch+1}/{EPOCHS} ---")
    for step, batch in enumerate(train_loader):
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        with torch.amp.autocast('cuda', enabled=torch.cuda.is_available()):
            out = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = out.loss
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        train_loss += loss.item()
        if (step + 1) % 50 == 0 or (step + 1) == len(train_loader):
            print(f"  Step [{step+1}/{len(train_loader)}] | Batch Loss: {loss.item():.4f}")

    val_loss, val_acc, val_prec, val_rec, val_f1, _, _ = evaluate(model, val_loader)
    print(f" Epoch {epoch+1} Results -> Val Loss: {val_loss:.4f} | Val Acc: {val_acc*100:.2f}% | Val Macro F1: {val_f1*100:.2f}%")
    if val_f1 > best_f1:
        best_f1 = val_f1
        print("  --> Saving best checkpoint to models/muril_cyberbullying_v2...")
        model.save_pretrained(output_dir)
        tokenizer.save_pretrained(output_dir)
        with open(os.path.join(output_dir, 'label_map.json'), 'w') as f:
            json.dump({'label_map': LABEL_MAP, 'id_to_label': ID_TO_LABEL}, f, indent=2)

## 5. Test Set Evaluation & Confusion Matrix

In [ ]:
test_loss, test_acc, test_prec, test_rec, test_f1, test_preds, test_labels = evaluate(model, test_loader)
print("=" * 70)
print("FINAL TEST SET REPORT (MuRIL v2 6-Class)")
print("=" * 70)
print(classification_report(test_labels, test_preds, target_names=CLASS_NAMES, digits=4))

cm = confusion_matrix(test_labels, test_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Confusion Matrix: MuRIL v2 (Combined English & Hinglish)', fontsize=14, fontweight='bold')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.tight_layout()
plt.show()